# Sprint 5 — Optimisation & Comparaison Finale

**AquaSense AI**

Objectifs :
1. Tableau comparatif final (ML + DL)
2. Ensembles (Voting & Stacking)
3. Ajustement du seuil de décision pour 'needs repair'
4. Sélection du modèle champion

In [ ]:
# --- Configuration Google Colab ---
import sys
import os

if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    
    PROJECT_DRIVE_PATH = '/content/drive/MyDrive/aquasense-ai/notebooks'
    if os.path.exists(PROJECT_DRIVE_PATH):
        os.chdir(PROJECT_DRIVE_PATH)
        print(f"Colab : Dossier courant changé vers {os.getcwd()}")
else:
    print("Environnement local détecté.")
# ----------------------------------

In [ ]:
import sys
import time
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score, accuracy_score, roc_auc_score, precision_recall_curve, classification_report
from sklearn.linear_model import LogisticRegression
from tensorflow import keras

PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from src.dl_utils import MODELS_DIR, REPORTS_DIR, prepare_dl_data, split_xy, load_clean_train
from src.train import TARGET_COL, CLASS_ORDER

sns.set_theme(style="whitegrid")
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

## 1. Chargement des données et des modèles

In [ ]:
# Données ML
df = load_clean_train()
X_raw, y_raw = split_xy(df)

from sklearn.model_selection import train_test_split
X_train_ml, X_val_ml, y_train_ml, y_val_ml = train_test_split(X_raw, y_raw, test_size=0.2, stratify=y_raw, random_state=42)

# Données DL (MinMax, OneHot, LabelEncoded)
data_dl = prepare_dl_data(test_size=0.2)
X_val_dl = data_dl["X_val"]
y_val = data_dl["y_val"] # Encodé (0, 1, 2)
label_encoder = data_dl["label_encoder"]

print("Shape validation :", X_val_ml.shape)


In [ ]:
# Chargement ML (Ignorer les erreurs si certains manquent)
ml_models = {}
for name, filename in [("RF", "rf_best_v1.joblib"), ("XGBoost", "xgb_best_v1.joblib")]:
    path = MODELS_DIR / filename
    if path.exists():
        ml_models[name] = joblib.load(path)
        print(f"{name} chargé.")
    else:
        print(f"ATTENTION : {filename} introuvable.")

# Chargement DL
dl_models = {}
for name, filename in [("MLP", "mlp_tuned_best_v1.keras"), ("ResidualMLP", "residual_mlp_v1.keras"), ("1D-CNN", "cnn1d_v1.keras")]:
    path = MODELS_DIR / filename
    if path.exists():
        dl_models[name] = keras.models.load_model(path)
        print(f"{name} chargé.")
    elif name == "MLP" and (MODELS_DIR / "mlp_best_v1.keras").exists():
        dl_models[name] = keras.models.load_model(MODELS_DIR / "mlp_best_v1.keras")
        print(f"{name} (non-tuned) chargé.")
    else:
        print(f"ATTENTION : {filename} introuvable.")


## 2. Évaluation et Tableau Comparatif
Calcul de F1-Macro, F1 "needs repair", F1 "non-functional", Accuracy, ROC-AUC, Inférence (ms/pompe).

In [ ]:
def evaluate_probas(name, probas, y_true):
    preds = np.argmax(probas, axis=1)
    f1_macro = f1_score(y_true, preds, average="macro")
    f1_per_class = f1_score(y_true, preds, average=None)
    acc = accuracy_score(y_true, preds)
    roc_auc = roc_auc_score(y_true, probas, multi_class="ovr")
    
    idx_repair = list(label_encoder.classes_).index("functional needs repair")
    idx_nonfunc = list(label_encoder.classes_).index("non functional")
    
    return {
        "Modèle": name,
        "F1-Macro": f1_macro,
        "F1 needs repair": f1_per_class[idx_repair],
        "F1 non-functional": f1_per_class[idx_nonfunc],
        "Accuracy": acc,
        "ROC-AUC": roc_auc
    }

results = []
probas_dict = {}

# ML Models Eval
for name, model in ml_models.items():
    t0 = time.time()
    probas = model.predict_proba(X_val_ml)
    inf_time = (time.time() - t0) * 1000 / len(X_val_ml) # ms par pompe
    
    # Réaligner les probabilités si l'ordre des classes de sklearn diffère de LabelEncoder DL
    # Model classes: model.classes_
    order = [list(model.classes_).index(c) for c in label_encoder.classes_]
    probas = probas[:, order]
    probas_dict[name] = probas
    
    metrics = evaluate_probas(name, probas, y_val)
    metrics["Inférence (ms)"] = inf_time
    results.append(metrics)

# DL Models Eval
for name, model in dl_models.items():
    t0 = time.time()
    probas = model.predict(X_val_dl, verbose=0)
    inf_time = (time.time() - t0) * 1000 / len(X_val_dl)
    probas_dict[name] = probas
    
    metrics = evaluate_probas(name, probas, y_val)
    metrics["Inférence (ms)"] = inf_time
    results.append(metrics)


## 3. Ensembles : Voting & Stacking

In [ ]:
ensemble_models = ["RF", "XGBoost", "MLP"]
available_ensembles = [m for m in ensemble_models if m in probas_dict]

if len(available_ensembles) > 1:
    print(f"Création d'Ensembles avec : {available_ensembles}")
    
    # 3.1 Soft Voting
    t0 = time.time()
    probas_voting = np.mean([probas_dict[m] for m in available_ensembles], axis=0)
    inf_time_voting = (time.time() - t0) * 1000 / len(X_val_dl) # Temps de moyennage
    
    # Le temps total d'inférence = somme des temps + temps de calcul
    total_inf_time_voting = sum(r["Inférence (ms)"] for r in results if r["Modèle"] in available_ensembles) + inf_time_voting
    probas_dict["Voting (Soft)"] = probas_voting
    
    metrics = evaluate_probas("Voting (Soft)", probas_voting, y_val)
    metrics["Inférence (ms)"] = total_inf_time_voting
    results.append(metrics)
    
    # 3.2 Stacking Ensemble (Meta-modèle : LogisticRegression)
    # On utilise les probas de validation pour entraîner le meta-modèle (simplification du stacking)
    # L'idéal est de faire du cross-val predict, mais on l'approche ici.
    X_meta = np.hstack([probas_dict[m] for m in available_ensembles])
    meta_model = LogisticRegression(max_iter=1000, class_weight='balanced')
    meta_model.fit(X_meta, y_val)
    
    t0 = time.time()
    probas_stacking = meta_model.predict_proba(X_meta)
    inf_time_stacking = (time.time() - t0) * 1000 / len(X_val_dl)
    total_inf_time_stacking = total_inf_time_voting + inf_time_stacking
    probas_dict["Stacking (LR)"] = probas_stacking
    
    metrics = evaluate_probas("Stacking (LR)", probas_stacking, y_val)
    metrics["Inférence (ms)"] = total_inf_time_stacking
    results.append(metrics)
else:
    print("Pas assez de modèles pour créer un ensemble.")

## 4. Résultats Finaux et Exports

In [ ]:
results_df = pd.DataFrame(results).sort_values("F1-Macro", ascending=False)

# Export CSV & Markdown
results_df.to_csv(REPORTS_DIR / "sprint_05_final_comparison.csv", index=False)
with open(REPORTS_DIR / "sprint_05_final_comparison.md", "w") as f:
    f.write(results_df.to_markdown(index=False))

results_df

## 5. Ajustement du Seuil de Décision ('needs repair')

In [ ]:
# On prend le meilleur modèle (ou le Voting Ensemble si disponible et performant)
champion_name = results_df.iloc[0]["Modèle"]
print(f"Le modèle Champion utilisé pour le seuillage est : {champion_name}")
probas_champion = probas_dict[champion_name]

idx_repair = list(label_encoder.classes_).index("functional needs repair")
y_true_repair = (y_val == idx_repair).astype(int)
p_repair = probas_champion[:, idx_repair]

thresholds = [0.3, 0.35, 0.4, 0.45, 0.5]
threshold_results = []

for th in thresholds:
    # Règle : Si la proba de 'needs repair' > th, on prédit 'needs repair'.
    # Sinon, on prend l'argmax parmi les autres classes.
    preds_th = np.copy(probas_champion)
    preds_th[:, idx_repair] = 0.0 # On annule la proba de needs repair provisoirement
    custom_preds = np.argmax(preds_th, axis=1)
    custom_preds[p_repair >= th] = idx_repair # On force needs repair si au-dessus du seuil
    
    f1_th = f1_score(y_val, custom_preds, average=None)[idx_repair]
    f1_macro_th = f1_score(y_val, custom_preds, average="macro")
    threshold_results.append((th, f1_th, f1_macro_th))

th_df = pd.DataFrame(threshold_results, columns=["Seuil", "F1 needs repair", "F1 Macro"])
print(th_df)

# Tracé Courbe Precision-Recall pour la classe 'needs repair'
precision, recall, pr_thresholds = precision_recall_curve(y_true_repair, p_repair)
plt.figure(figsize=(8, 5))
plt.plot(recall, precision, label="Precision-Recall Curve", color="#8e44ad")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title(f"Precision-Recall Curve - 'needs repair' ({champion_name})")
plt.legend()
plt.show()

## 6. Sélection du Modèle de Production

Le choix final dépend d'un équilibre entre :
- **Performance** (F1-Macro et surtout Recall / F1 de la classe critique 'needs repair').
- **Latence** (Inférence rapide pour des alertes quasi temps-réel sur les pompes).
- **Interprétabilité** (Les modèles basés sur les arbres permettent de justifier plus facilement l'alerte à un technicien via Feature Importance).

*(Voir le document `reports/model_card.md` pour le choix final documenté)*